# E09 — Diagnóstico de caminhos da rede BRO-S

Importação da rede instalada, inspeção de caminhos e extração de features. Requer SDK 0.1.47 com extras notebook. Abra na pasta notebooks do workspace. As seções 1 a 3 são locais; as seções 4 e 5 precisam dos serviços. A seção 6 associa as métricas por salto às features; as seções 7 e 8 aplicam o modelo e mostram o diagnóstico dos caminhos.

In [ ]:
from cisei_planning_sdk import InstalledNetworkProject

project = InstalledNetworkProject("../projects/e09_diagnostico_bro")

## 1. Importar a topologia instalada

O SDK lê os perfis de scenario.toml, as 25 instâncias de nodes.csv e os 24 enlaces explícitos. As conexões existentes são preservadas. O acesso ao backbone é BRO-S-P7-001. O campo next_hop é informativo; a conectividade vem dos enlaces. A importação registra os pais no result.json sem executar planejamento nem atribuir um rank de qualidade.

Esta primeira versão aceita uma floresta com um acesso ao backbone por componente. Referências inválidas, ciclos e componentes sem acesso ou com acessos ambíguos são rejeitados, em vez de gerar rotas novas.

In [ ]:
project.import_topology();

## 2. Inspecionar a topologia lógica

Selecione um equipamento para destacar os enlaces até o backbone e consultar o número de saltos. Azul identifica acesso ao backbone, verde identifica os demais equipamentos e laranja identifica o caminho selecionado. Essas cores não representam desempenho. A disposição hierárquica facilita a inspeção de pais e dependências.

In [ ]:
project.show_topology(layout="logical");

## 3. Localizar o caminho geograficamente

A mesma rede pode ser inspecionada pelas coordenadas projetadas, sem GeoJSON e sem mapa de fundo. A seleção também destaca o caminho até o backbone. As duas vistas usam a mesma topologia importada; selecionar um nó em uma vista não altera a outra.

In [ ]:
project.show_topology(layout="geographic");

## 4. Extrair as features dos enlaces instalados

O SDK reutiliza a extração em lotes do serviço de planejamento, exclusivamente para as 24 conexões existentes. As requisições seguem o sentido original pai → filho, compatível com o RSSI que será associado depois. As duas conexões back-to-back, identificadas na origem como b2b e sem distância geográfica, permanecem na topologia como internas: são extraídos somente os 22 trechos de rádio. O cache é salvo em features.json e reaproveitado nas execuções seguintes. O result.json da topologia instalada é preservado.

A etapa existente do serviço também calcula os custos configurados, mas esta chamada não seleciona rotas, não atribui rank aos caminhos e não compara medições. hop_metrics.json e path_metrics.json ainda não são consumidos.

In [ ]:
from cisei_planning_sdk import PlanningClient

client = PlanningClient("http://network-service:8080", workspace="..", timeout=1900.0)
project.extract_features(client, geo_base_url="http://features:8080", batch_size=10);

## 5. Inspecionar os enlaces

O painel reutiliza o visualizador de features existente: selecione um equipamento ou enlace e escolha o perfil, os mapas de terreno/cobertura, edificações ou as próprias features. Pressione Show para preparar o primeiro enlace; a construção do painel não solicita dados geográficos. Seleções seguintes atualizam a inspeção.

A sessão de inspeção é independente do cache de extração em lotes. Cada inspeção prepara apenas o enlace selecionado, com as coordenadas, alturas e frequência do projeto, sem recalcular a topologia.

In [ ]:
from cisei_features_sdk.notebook import FeaturesNotebook

geo = FeaturesNotebook("http://features:8080", user=None)
inspector = project.inspect_links(geo);

## 6. Enriquecer as features com as métricas por salto

A chamada mantém o enriquecimento da E08: gera enriched_features.json com observations, preservando o features.json como cache puro, sem extrair novamente as características. baseline.json indica hop_metrics.json como entrada. Cada observação identifica seu link_id, transmissor e receptor no sentido pai → filho.

O SDK valida os identificadores e o sentido das conexões antes de associar as 22 observações de rádio. Os dois registros das conexões internas permanecem no arquivo de entrada e são excluídos explicitamente do enriquecimento: seu RSSI zero não é interpretado como sinal recebido. Repetir a chamada não duplica observações; alterações de observações existentes exigem replace=True. A topologia e o modelo treinado da E08 são preservados. path_metrics.json continua separado para cruzamento com a aplicação.

In [ ]:
project.enrich_features();

## 7. Aplicar o modelo da E08 e classificar os enlaces

O SDK lê o modelo já treinado e enriched_features.json, compara previsão e medição e grava evaluation.json. Não há novo treinamento nem seleção de caminhos. O limiar de -80 dBm é ilustrativo e deve ser definido pelo engenheiro.

Verde: atendimento satisfatório (previsão e medição boas). Vermelho: investigar instalação ou defeito (previsão boa e medição ruim). Azul: medição satisfatória; sem correção (previsão ruim e medição boa). Marrom: revisar projeto (previsão e medição ruins). As duas conexões internas não recebem classificação de RSSI. Sem medição, a previsão pode indicar revisão de projeto; uma previsão satisfatória permanece identificada como sem medição.

In [ ]:
project.evaluate_links(rssi_threshold_dbm=-80);

## 8. Inspecionar o diagnóstico dos caminhos

As cores dos enlaces seguem a E08 nas vistas lógica e geográfica. Selecione um nó para destacar seu caminho até o backbone, mantendo as cores de classificação. O painel resume quantos enlaces precisam de inspeção e revisão de projeto e apresenta a previsão e a medição de cada trecho. Clicar diretamente em um enlace mostra seu diagnóstico individual. As conexões por cabo aparecem tracejadas em cinza na vista lógica; no mapa, as posições coincidentes podem sobrepor esses trechos.

O diagnóstico localiza segmentos que merecem investigação; a indicação de inspeção não comprova um defeito físico específico. Os nós mantêm cor neutra: a classificação é do enlace. As métricas de caminho e a aplicação ainda não são cruzadas nesta revisão.

In [ ]:
project.show_topology(layout="logical", diagnosis=True);
project.show_topology(layout="geographic", diagnosis=True);